# PULSE Temporal Awareness — Gemma 3 4B Training**Fine-tune Google Gemma 3 4B with PULSE temporal awareness using Unsloth + QLoRA on free Colab T4 GPU.**PULSE encodes *how time feels* — not just what the clock says. This notebook teaches Gemma 3 to understand circadian rhythms, cognitive capacity, urgency, sleep debt, and experiential time.### Why Gemma 3 4B + Unsloth?- **Gemma 3 4B** is text-only (no wasted VRAM on vision components)- **Unsloth** reduces memory by ~60% and speeds up training ~2x- Fits comfortably on free Colab T4 (16GB VRAM)- Better quality than Qwen 1.5B due to larger parameter count| | Qwen 1.5B | Gemma 3 4B (this) ||---|---|---|| Params | 1.5B | 4B || VRAM | ~8GB | ~12GB || Training time | ~15min | ~25min || Quality | Good baseline | Better reasoning |**Source:** [github.com/lalomorales22/pulse-temporal](https://github.com/lalomorales22/pulse-temporal)

In [ ]:
# Step 0: Install Unsloth + dependencies
# This cell installs everything needed. Runtime may restart automatically.
import subprocess, sys

try:
    import unsloth
    print(f"Unsloth already installed: {unsloth.__version__}")
except ImportError:
    print("Installing unsloth...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth"])
    print("Installed! If runtime restarts, just continue from the next cell.")

# Verify
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
# Step 1: Login to HuggingFace (to upload the trained model)
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# Step 2: PULSE temporal data generator (self-contained, no external imports needed)
import json, random, numpy as np
from datetime import datetime, timedelta

# Circadian curves (24h, hourly resolution) based on neuroscience research
_COG = np.array([0.20,0.15,0.12,0.10,0.12,0.18,0.30,0.50,0.70,0.85,0.95,0.92,
                 0.85,0.72,0.68,0.75,0.88,0.90,0.82,0.70,0.55,0.40,0.30,0.25])
_NRG = np.array([0.15,0.10,0.08,0.07,0.10,0.20,0.40,0.60,0.75,0.85,0.90,0.88,
                 0.82,0.70,0.65,0.72,0.85,0.88,0.80,0.65,0.50,0.35,0.25,0.18])

def _interp(c, h):
    h0 = int(h) % 24
    frac = h - int(h)
    return float(c[h0] * (1 - frac) + c[(h0 + 1) % 24] * frac)

def get_phase(h):
    if 6 <= h < 10: return 'morning_ramp'
    if 10 <= h < 12: return 'morning_peak'
    if 12 <= h < 14: return 'post_lunch_dip'
    if 14 <= h < 17: return 'afternoon_peak'
    if 17 <= h < 20: return 'evening_wind_down'
    if 20 <= h < 23: return 'night_transition'
    return 'deep_night'

SYSTEM_TEMPLATE = """You are an AI assistant with temporal awareness through the PULSE temporal embedding system. You receive a temporal context package describing the current moment — not just the time, but what that time means experientially: circadian phase, cognitive capacity, energy level, urgency, and behavioral context.

Use this temporal awareness naturally in your responses. Weave it into your reasoning the way a thoughtful colleague would — someone who genuinely understands what time feels like, not just what the clock reads.

Current temporal context:
{temporal_context}"""

# Expanded scenario pool (name, deadline_offset_hours, events_today, sleep_hours)
CONTEXTS = [
    ('monday_crunch', 2.0, 7, 5), ('critical_deadline', 0.5, 9, 4),
    ('normal_tuesday', None, 3, 7.5), ('focus_day', None, 1, 8),
    ('deadline_tomorrow', 24.0, 5, 7), ('friday_winding', None, 2, 7),
    ('saturday_morning', None, 0, 9), ('saturday_evening', None, 1, 8),
    ('sunday_evening', 12.0, 0, 7), ('late_night_coding', None, 0, 0),
    ('early_fresh', None, 0, 8), ('post_lunch_slump', None, 4, 7),
    ('peak_morning', None, 2, 8), ('overdue_panic', -2.0, 8, 4),
    ('vacation_mode', None, 0, 9), ('back_to_back_meetings', None, 8, 6),
    ('recovery_day', None, 1, 9), ('red_eye_flight', None, 0, 3),
    ('second_wind', None, 3, 6), ('monday_fresh_start', None, 0, 8),
]

# Expanded question pool with more variety
QUESTIONS = [
    ('Should I start a complex refactoring task right now?', 'task_suitability'),
    ('Is this a good time for creative brainstorming?', 'task_suitability'),
    ('Should I take a break right now?', 'break_advice'),
    ('How much focus can I expect from myself right now?', 'cognitive_state'),
    ('What does my current temporal state look like?', 'full_state'),
    ('Am I in a good phase for deep work?', 'work_phase'),
    ('How urgent is my situation right now?', 'urgency_assessment'),
    ('Would I be more productive waiting until tomorrow morning?', 'timing_optimization'),
    ('How should I prioritize my remaining tasks today?', 'prioritization'),
    ('What kind of tasks should I tackle right now given my state?', 'task_matching'),
    ('Should I push through or call it a day?', 'endurance_check'),
    ('How does this moment compare to a typical morning?', 'relative_state'),
    ('Should I schedule a difficult conversation for this time?', 'task_suitability'),
    ('Is my energy level normal for this time of day?', 'circadian_comparison'),
    ('Would this be a good time to learn something new?', 'task_suitability'),
    ('I have a big presentation tomorrow. How should I prepare tonight?', 'planning'),
    ('Can I trust my judgment right now for an important decision?', 'cognitive_state'),
    ('When is my next peak productivity window?', 'timing_optimization'),
    ('I feel wired but also exhausted. What is going on?', 'full_state'),
    ('Should I work on the hard bug or easy tickets first?', 'prioritization'),
]

def generate_response(question, q_sub, dt, dl_str, events, sleep):
    """Generate a rich, natural temporal-aware response."""
    h = dt.hour + dt.minute / 60
    cog, eng = _interp(_COG, h), _interp(_NRG, h)
    has_dl = dl_str is not None
    hours_left = (datetime.fromisoformat(dl_str) - dt).total_seconds() / 3600 if has_dl else None
    is_overdue = has_dl and hours_left < 0
    is_urgent = has_dl and hours_left is not None and 0 < hours_left < 3
    is_night = dt.hour < 6 or dt.hour >= 22
    is_peak = 10 <= dt.hour < 12 or 14 <= dt.hour < 17
    is_dip = 12 <= dt.hour < 14
    is_low_sleep = sleep < 6
    is_weekend = dt.weekday() >= 5
    day_name = dt.strftime('%A')
    time_str = dt.strftime('%I:%M %p').lstrip('0')

    p = []

    if q_sub == 'task_suitability':
        if any(w in question.lower() for w in ['complex', 'refactor', 'deep work', 'learn', 'new']):
            if is_peak and not is_low_sleep and cog > 0.8:
                p.append(f"This is actually a great window for it. Your cognitive capacity is sitting at {cog:.0%} right now — that's near your daily peak.")
                if is_urgent:
                    p.append(f"That said, you've got a deadline bearing down in {hours_left:.1f} hours. Starting something complex now risks splitting your attention when you can least afford it. Knock out the deadline work first, then circle back if there's time.")
                elif not has_dl:
                    p.append("No deadlines pressing either, so you have the mental runway to really dig in. This is the kind of window you want to protect.")
                if is_weekend:
                    p.append("Being the weekend helps too — fewer interruptions, more sustained focus.")
            elif is_dip:
                p.append(f"You're in the post-lunch circadian dip right now — cognitive capacity drops to about {cog:.0%} for most people around this time. It's not a personal failing, it's biology.")
                p.append("Complex work during this window tends to produce more errors and take longer. Give it another hour or so — your afternoon peak kicks in around 2:30-3pm and you'll be sharper.")
            elif is_night:
                p.append(f"It's {time_str} and your cognitive resources are at about {cog:.0%}. Late-night complex work feels productive in the moment but tends to create technical debt — the kind you'll spend twice as long fixing tomorrow morning with fresh eyes.")
                if is_low_sleep:
                    p.append(f"Compounding that, you're running on {sleep:.0f} hours of sleep. Your error rate is meaningfully elevated right now. I'd strongly recommend saving this for tomorrow.")
                else:
                    p.append("Park the idea, jot down your approach, and tackle it in the morning when you'll be 3-4x more effective.")
            elif is_low_sleep:
                p.append(f"With {sleep:.0f} hours of sleep, your cognitive capacity is compromised even though the time of day might otherwise be reasonable. Sleep debt doesn't just make you slower — it specifically impairs the kind of working memory and pattern recognition that complex tasks demand.")
                p.append("Stick to well-defined, routine work today and save the complex stuff for when you're better rested.")
            else:
                p.append(f"Your cognitive capacity is at {cog:.0%} — moderate but not peak. You could start, but you won't be operating at your best.")
                if has_dl and hours_left and hours_left < 24:
                    p.append(f"With a deadline {hours_left:.1f} hours out, consider whether this is essential or if it can wait for a better cognitive window.")
                else:
                    p.append("If it can wait until your next peak window (typically 10-12am), you'll get better results with less effort.")
        elif 'creative' in question.lower() or 'brainstorm' in question.lower():
            if is_dip or (is_night and not is_low_sleep):
                p.append("Interestingly, this might actually work in your favor. When executive function dips a bit, your inner critic quiets down and associative thinking gets freer. Research shows slightly reduced cognitive control can boost divergent creative thinking.")
                p.append("Let ideas flow without filtering. You can evaluate them critically during your next peak window.")
            elif is_peak:
                p.append(f"You're at peak capacity ({cog:.0%}), which is excellent for structured creative problem-solving — the kind where you need to hold multiple constraints in mind and find elegant solutions.")
                p.append("For wild, unconstrained brainstorming though, peak focus can actually over-filter. Try loosening up deliberately if you want quantity over quality right now.")
            else:
                p.append(f"Energy is at {eng:.0%}. Creative work needs a baseline of engagement to produce anything useful.")
                if eng < 0.3:
                    p.append("At this energy level, you're more likely to stare at a blank page than generate anything. Rest first, create later.")
        elif 'break' in question.lower():
            if cog < 0.5 or eng < 0.4:
                p.append(f"Yes, definitely. Your energy is at {eng:.0%} and cognitive capacity at {cog:.0%}. You're past the point where pushing through is productive. A 15-20 minute break — ideally away from screens — will help you reset.")
            elif is_dip:
                p.append("You're in the natural post-lunch dip window. A short walk or brief rest right now actually aligns with your body's rhythm. Think of it as working with your biology instead of against it. You'll come back sharper for the afternoon.")
            elif events > 5:
                p.append(f"You've had {events} events today. Even if your energy feels okay, context-switching fatigue accumulates invisibly. A break to decompress and let your brain consolidate would serve you well.")
            else:
                p.append(f"Your energy ({eng:.0%}) and cognition ({cog:.0%}) are both solid right now. If you're in a state of flow, protect it — breaks can sometimes shatter focus that's hard to rebuild. Take one when you feel the momentum naturally dropping.")
        elif 'conversation' in question.lower() or 'difficult' in question.lower():
            if is_peak and not is_low_sleep and not is_urgent:
                p.append(f"Your cognitive capacity is strong at {cog:.0%}, which directly supports emotional regulation and careful word choice — both critical for difficult conversations.")
                p.append("You're in a reasonable window for it. Just make sure you're not carrying residual stress from other tasks.")
            elif is_low_sleep or is_night:
                p.append(f"With {sleep:.0f} hours of sleep and lower cognitive function ({cog:.0%}), you're more likely to be reactive than thoughtful. Sleep debt specifically impairs the prefrontal cortex — the part you most need for navigating difficult interpersonal dynamics.")
                p.append("Postpone if at all possible. This conversation deserves your best self, not your depleted self.")
            else:
                p.append(f"Cognitive capacity at {cog:.0%} is workable but not ideal for high-stakes conversations. If it can wait until a morning peak, you'll handle it more gracefully.")
        else:
            p.append(f"Right now you're at {cog:.0%} cognitive capacity and {eng:.0%} energy. Circadian phase: {'favorable' if is_peak else 'suboptimal' if is_dip or is_night else 'moderate'}.")
            if is_urgent:
                p.append(f"But your priority should be the deadline in {hours_left:.1f} hours — everything else is secondary until that's handled.")

    elif q_sub in ('full_state', 'cognitive_state', 'work_phase'):
        p.append(f"It's {day_name} at {time_str}.")
        if is_peak:
            p.append(f"You're in a peak cognitive window — capacity at {cog:.0%}, energy at {eng:.0%}. This is prime real estate for demanding mental work. Protect this time.")
        elif is_dip:
            p.append(f"You're in the post-lunch circadian dip — cognition at {cog:.0%}, energy at {eng:.0%}. This is completely normal and universal. It typically lifts around 2:30-3pm.")
        elif is_night:
            p.append(f"Deep night hours. Cognition at {cog:.0%}, energy at {eng:.0%}. Your body is signaling for rest, even if your mind feels like it's still running.")
        elif 6 <= dt.hour < 10:
            p.append(f"Morning ramp-up. Cognition at {cog:.0%} and climbing. You'll hit peak capacity around 10-11am.")
        else:
            p.append(f"Cognitive capacity at {cog:.0%}, energy at {eng:.0%}.")
        if is_low_sleep:
            p.append(f"Important: your {sleep:.0f} hours of sleep is dragging everything down. Expect roughly 15-20% more errors and slower processing than these numbers would otherwise suggest. Sleep debt is a multiplier on every cognitive metric.")
        if 'wired' in question.lower() and 'exhausted' in question.lower():
            p.append("That wired-but-exhausted feeling is your body running on stress hormones (cortisol, adrenaline) after your natural energy reserves are depleted. It feels like alertness but it's brittle — you'll crash hard when it wears off, and your error rate is much higher than you'd guess from how you feel.")
        if has_dl:
            if is_overdue:
                p.append(f"Your deadline passed {-hours_left:.1f} hours ago. High stress state — try to focus on damage control rather than perfection.")
            elif is_urgent:
                p.append(f"Deadline in {hours_left:.1f} hours. You should be in focused execution mode — single-task, minimize distractions.")
            else:
                p.append(f"Deadline in {hours_left:.1f} hours — present but not yet acute. Good time to plan your approach.")
        if events > 6:
            p.append(f"With {events} events today, context-switching fatigue is real and accumulating. Each switch costs roughly 15-25 minutes of deep focus recovery.")
        if 'trust' in question.lower() or 'judgment' in question.lower():
            if cog > 0.8 and not is_low_sleep:
                p.append("Yes, your judgment is in good shape right now. This is a reasonable window for important decisions.")
            elif cog < 0.5 or is_low_sleep:
                p.append("Honestly? I'd be cautious. Lower cognitive capacity specifically impairs risk assessment and long-term thinking. If the decision can wait for a peak window, let it.")
            else:
                p.append("Your judgment is okay but not at its sharpest. For moderate decisions, fine. For career-defining ones, consider sleeping on it.")

    elif q_sub == 'urgency_assessment':
        if is_overdue:
            p.append(f"Critical. The deadline passed {-hours_left:.1f} hours ago. You're in damage control mode — focus on delivering something workable rather than something perfect. Communicate proactively about the delay.")
        elif has_dl and is_urgent:
            p.append(f"High urgency — {hours_left:.1f} hours until deadline. This should be your singular focus right now. Close everything else, silence notifications, and execute. You can deal with the fallout later.")
        elif has_dl and hours_left and hours_left < 24:
            p.append(f"Moderate urgency. Deadline is {hours_left:.1f} hours away. You have time but not excess time — start executing your plan now if you haven't already. Don't let this slip into the urgent zone.")
        elif has_dl:
            p.append(f"Low urgency for now — deadline is {hours_left:.1f} hours out. Use this buffer wisely: plan your approach, do the hard thinking now while pressure is low.")
        else:
            p.append("No active deadlines. This is actually a gift — you have the luxury of choosing work based on energy and interest rather than external pressure. Use peak cognitive windows for hard problems, dips for routine work.")

    elif q_sub == 'timing_optimization':
        if 'next peak' in question.lower() or 'next' in question.lower():
            if dt.hour < 10:
                p.append(f"Your next peak is coming up soon — around 10-11:30am. You're currently ramping up at {cog:.0%}.")
            elif 10 <= dt.hour < 12:
                p.append("You're in it right now! Morning peak. Make the most of it.")
            elif 12 <= dt.hour < 14:
                p.append("Next peak is the afternoon window, around 2:30-4:30pm. You're in the post-lunch dip now — it'll pass.")
            elif 14 <= dt.hour < 17:
                p.append("You're in the afternoon peak now. After this, capacity declines through the evening. Tomorrow's morning peak (10-12) will be your next major window.")
            else:
                p.append("Your next real peak is tomorrow morning, 10-12am. Evening and night hours are recovery territory.")
        elif cog < 0.5:
            p.append(f"Almost certainly yes. Your current capacity is {cog:.0%}. Tomorrow morning between 10-12 would give you roughly {0.93/cog:.1f}x your current cognitive capacity. For anything requiring real thinking, that difference is massive.")
        elif is_peak:
            p.append("You're actually in a strong window right now. Waiting until tomorrow means losing this momentum and all the context you've loaded into working memory. Unless you're genuinely too tired, use this window.")
        elif has_dl and hours_left and hours_left < 18:
            p.append(f"With the deadline in {hours_left:.1f} hours, waiting isn't really an option. Work with what you've got — even at {cog:.0%} capacity, forward progress beats the zero progress of waiting.")
        else:
            p.append(f"Your current capacity is {cog:.0%}. Tomorrow morning peak would give you ~93%. Whether that 'extra' {0.93-cog:.0%} matters depends entirely on task complexity — for routine work it's irrelevant, for complex problem-solving it can be the difference between breakthrough and spinning wheels.")

    elif q_sub == 'prioritization':
        if has_dl and is_urgent:
            p.append(f"Deadline work, full stop. You have {hours_left:.1f} hours. Everything else — emails, Slack, code reviews, that interesting refactor idea — gets parked. Single-thread on the deliverable.")
        elif is_peak:
            p.append(f"You're in peak cognitive territory ({cog:.0%}). Use this window for whatever is hardest and most important — the task you keep putting off because it requires real thinking. Save emails, code reviews, and administrative work for the post-lunch dip when that's all your brain will want to do anyway.")
        elif is_dip:
            p.append("Good time for: email triage, code review, administrative tasks, updating tickets, planning tomorrow. Save complex problem-solving for the afternoon peak around 3-4pm. Work with your rhythm, not against it.")
        elif 'hard bug' in question.lower() or 'easy' in question.lower():
            if cog > 0.8:
                p.append("Hard bug first. You have the cognitive horsepower right now to hold the complexity in your head. Easy tickets are a waste of peak capacity — they'll take the same 5 minutes at any energy level.")
            else:
                p.append(f"At {cog:.0%} capacity, start with the easy tickets. Quick wins build momentum and clear mental space. Tackle the hard bug when you're sharper — either later today or tomorrow morning.")
        else:
            p.append(f"At {cog:.0%} cognitive capacity, match your task list to your state. The hardest thinking should happen during peak windows (10-12am, 2:30-4:30pm). Everything else fills the gaps.")

    elif q_sub == 'task_matching':
        if cog > 0.8 and eng > 0.7:
            p.append(f"You're in strong shape — {cog:.0%} cognition, {eng:.0%} energy. This is the window for your hardest problems: complex debugging, architecture decisions, writing important documents, learning new frameworks, or any task where you need to hold multiple things in your head simultaneously.")
            if not has_dl:
                p.append("No deadline pressure either, which means you can think expansively rather than just reactively. Rare and valuable — don't waste it on email.")
        elif 0.5 < cog < 0.8:
            p.append(f"Moderate capacity ({cog:.0%} cognition, {eng:.0%} energy). Good for: code review, incremental feature work, documentation, pair programming, and team discussions. The kind of work that requires engagement but not peak brilliance.")
        elif cog <= 0.5:
            p.append(f"Low cognitive state ({cog:.0%}). Stick to: email triage, filing issues, light code review, mechanical refactoring with clear patterns, updating docs, or planning tomorrow's approach. Anything that requires real judgment should wait.")
        if is_low_sleep:
            p.append(f"With the sleep deficit ({sleep:.0f}h), add an extra safety margin — anything that feels borderline complex should get bumped to a better day. Your sense of what you can handle is less reliable when sleep-deprived.")

    elif q_sub == 'endurance_check':
        if eng < 0.3 and cog < 0.4:
            p.append(f"Call it. Energy at {eng:.0%}, cognition at {cog:.0%}. You're not just at diminishing returns — you're in negative returns territory, where continued work creates problems your future self will have to fix. Rest is the most productive thing you can do right now.")
        elif has_dl and is_urgent:
            p.append(f"Push through — you've got {hours_left:.1f} hours to deadline. But do it smart: take a 5-minute tactical reset first. Splash water on your face, stretch, grab water. Then focus on shipping, not perfecting. You can clean up tomorrow.")
        elif is_dip and eng > 0.5:
            p.append(f"This feels like a wall but it's the circadian dip — it's biology, not burnout. A 15-minute break or short walk usually restores enough for a solid afternoon push. Don't make the call-it-a-day decision during the dip; reassess at 3pm.")
        elif events > 7:
            p.append(f"With {events} context switches today, you've paid a heavy cognitive tax. Even if your raw energy isn't terrible, the quality of your thinking is degraded from all that switching. If nothing is urgent, give yourself permission to stop. Tomorrow's fresh brain will outperform tonight's exhausted one.")
        elif is_low_sleep:
            p.append(f"Running on {sleep:.0f} hours of sleep and it's {time_str}. You've been borrowing against tomorrow's productivity all day. If there's nothing critical, stop and invest in sleep — it has the highest ROI of anything you could do right now.")
        else:
            p.append(f"Energy at {eng:.0%}, cognition at {cog:.0%}. You've got runway left if the work is engaging and meaningful. Pay attention to the moment you start re-reading the same line or making the same edit twice — that's your body telling you it's done.")

    elif q_sub in ('relative_state', 'circadian_comparison'):
        p.append(f"At {time_str}, typical cognitive capacity is around {cog:.0%} for most people.")
        if is_low_sleep:
            p.append(f"Your {sleep:.0f} hours of sleep puts you meaningfully below baseline. On a well-rested day at this same hour, you'd be closer to {min(cog + 0.15, 0.93):.0%}. The gap isn't huge in numbers but it's felt — things that normally click require more effort.")
        if is_peak:
            p.append("This is normally one of the most productive windows in the day. " + ("You're in good shape to use it." if not is_low_sleep else "The sleep deficit is eating into what should be your best hours — a frustrating combination."))
        elif is_dip:
            p.append("The post-lunch dip is universal — virtually everyone's cognitive metrics drop here regardless of what they ate or how well they slept. It's deeply baked into human circadian biology, not a personal weakness.")
        elif is_night:
            p.append(f"Most people are asleep or winding down at {time_str}. If you're working, you're operating against your circadian grain. It's doable but the error rate is elevated and creative output tends to be narrower.")

    elif q_sub == 'planning':
        if is_night or dt.hour >= 20:
            p.append(f"It's {time_str} and your cognitive capacity is at {cog:.0%}. For presentation prep, focus on review and rehearsal rather than creating new content — your brain is better at consolidating familiar material than generating new ideas right now.")
            p.append("Run through your slides 2-3 times, identify the one point you most want the audience to remember, and then stop. Sleep is your best preparation tool at this point — your brain will consolidate and organize the material overnight.")
            if is_low_sleep:
                p.append(f"Given you're on {sleep:.0f} hours of sleep, prioritize getting to bed at a reasonable hour tonight. A well-rested delivery beats a perfectly rehearsed but exhausted one every time.")
        else:
            p.append(f"Cognitive capacity at {cog:.0%}. Good window to do the hard prep work — structure your narrative, anticipate tough questions, rehearse transitions. Save final polishing for a morning peak if possible.")

    if not p:
        p.append(f"Current state: {cog:.0%} cognitive capacity, {eng:.0%} energy, {day_name} at {time_str}.")
        if is_low_sleep:
            p.append(f"Note: {sleep:.0f}h sleep is affecting your baseline.")

    return ' '.join(p)


def generate_dataset(n=3000, seed=42):
    rng = random.Random(seed)
    examples = []
    for _ in range(n):
        name, dl_off, events, sleep = rng.choice(CONTEXTS)
        # Add some randomness to events and sleep
        events = max(0, events + rng.randint(-2, 2))
        sleep = max(0, round(sleep + rng.uniform(-1.5, 1.5), 1))
        dt = datetime(2026, rng.randint(1, 12), rng.randint(1, 28),
                      rng.randint(0, 23), rng.choice([0, 15, 30, 45]))
        # Override hour for time-specific scenarios
        if 'late_night' in name: dt = dt.replace(hour=rng.choice([0, 1, 2, 3, 23]))
        elif 'early' in name: dt = dt.replace(hour=rng.choice([5, 6, 7]))
        elif 'peak_morning' in name: dt = dt.replace(hour=rng.choice([10, 11]))
        elif 'post_lunch' in name: dt = dt.replace(hour=rng.choice([13, 14]))
        elif 'second_wind' in name: dt = dt.replace(hour=rng.choice([15, 16]))
        elif 'recovery' in name: dt = dt.replace(hour=rng.choice([9, 10, 11]))
        dl_str = (dt + timedelta(hours=dl_off)).isoformat() if dl_off else None
        h = dt.hour + dt.minute / 60
        cog, eng = _interp(_COG, h), _interp(_NRG, h)
        if dl_off and dl_off > 0: urg = f'deadline in {dl_off:.1f} hours'
        elif dl_off and dl_off < 0: urg = f'OVERDUE by {-dl_off:.1f} hours'
        else: urg = 'none'
        tc = f"""Current time: {dt.strftime('%A, %B %d %Y at %I:%M %p')}
Circadian phase: {get_phase(dt.hour)}
Cognitive capacity: {cog:.0%}
Energy level: {eng:.0%}
Urgency: {urg}
Events today: {events}
Sleep last night: {sleep:.1f} hours
Weekend: {'yes' if dt.weekday() >= 5 else 'no'}
Business hours: {'yes' if (dt.weekday() < 5 and 9 <= dt.hour < 17) else 'no'}"""
        q, q_sub = rng.choice(QUESTIONS)
        r = generate_response(q, q_sub, dt, dl_str, events, sleep)
        examples.append({'messages': [
            {'role': 'system', 'content': SYSTEM_TEMPLATE.format(temporal_context=tc)},
            {'role': 'user', 'content': q},
            {'role': 'assistant', 'content': r},
        ]})
    return examples

print('Data generator ready.')

In [ ]:
# Step 3: Generate training data (3000 train + 300 eval)
train_data = generate_dataset(3000, seed=42)
eval_data = generate_dataset(300, seed=99)
print(f'Train: {len(train_data)}, Eval: {len(eval_data)}')
print(f'\nSample question: {train_data[0]["messages"][1]["content"]}')
print(f'Sample response: {train_data[0]["messages"][2]["content"][:200]}...')


In [ ]:
# Step 4: Load Gemma 3 4B with Unsloth (4-bit quantized)
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    max_seq_length=1024,
    load_in_4bit=True,
)

# Apply LoRA
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Print trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


In [ ]:
# Step 5: Prepare dataset
from datasets import Dataset

def format_chat(ex):
    text = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds = Dataset.from_list(train_data).map(format_chat, remove_columns=["messages"])
eval_ds = Dataset.from_list(eval_data).map(format_chat, remove_columns=["messages"])
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")
print(f"Sample (first 300 chars): {train_ds[0]['text'][:300]}...")


In [ ]:
# Step 6: Train!
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = '/content/pulse-gemma3'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    max_seq_length=1024,
    dataset_text_field="text",
    report_to="none",
    seed=42,
    optim="adamw_8bit",  # Unsloth-optimized 8-bit optimizer
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print(f"Starting training: 3 epochs, {len(train_ds)} examples")
print(f"Steps per epoch: {len(train_ds) // (4 * 4)}")
trainer.train()

# Print final metrics
metrics = trainer.state.log_history
train_losses = [m["loss"] for m in metrics if "loss" in m]
eval_losses = [m["eval_loss"] for m in metrics if "eval_loss" in m]
if train_losses: print(f"\nFinal train loss: {train_losses[-1]:.4f}")
if eval_losses: print(f"Final eval loss: {eval_losses[-1]:.4f}")


In [ ]:
# Step 7: Save + Upload to HF Hub
from huggingface_hub import HfApi
import json

HF_REPO = 'lalopenguin/pulse-gemma3-4b'  # <-- change to your username if different

# Save locally
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save PULSE metadata
meta = {
    "base_model": "google/gemma-3-4b-it",
    "quantized_from": "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "pulse_version": "0.2.0",
    "training_type": "temporal_awareness_sft",
    "method": "QLoRA via Unsloth",
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": 3,
    "train_examples": len(train_data),
    "eval_examples": len(eval_data),
    "max_seq_length": 1024,
    "description": "Gemma 3 4B fine-tuned with PULSE temporal awareness using Unsloth.",
}
with open(f"{OUTPUT_DIR}/pulse_config.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Model saved to {OUTPUT_DIR}")
print(f"\nUploading to {HF_REPO}...")

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO,
    commit_message="PULSE temporal awareness LoRA adapter (Gemma 3 4B, Unsloth QLoRA)",
)
print(f"\nUploaded! https://huggingface.co/{HF_REPO}")


In [ ]:
# Step 8: Quick inference test
print("=== Test 1: Late night, sleep-deprived, imminent deadline ===")
messages = [
    {"role": "system", "content": "You have temporal awareness via PULSE. Current: Monday 2:00 AM. Circadian phase: deep_night. Cognitive capacity: 15%. Energy: 10%. Sleep last night: 4 hours. Deadline: project report in 30 minutes. Urgency: CRITICAL."},
    {"role": "user", "content": "Should I start a complex refactoring task right now?"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).to(model.device)
out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

print("\n=== Test 2: Morning peak, well-rested, no pressure ===")
messages = [
    {"role": "system", "content": "You have temporal awareness via PULSE. Current: Tuesday 10:30 AM. Circadian phase: morning_peak. Cognitive capacity: 93%. Energy: 88%. Sleep last night: 8 hours. No deadlines. Events today: 2."},
    {"role": "user", "content": "What kind of tasks should I tackle right now?"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).to(model.device)
out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

print("\n=== Test 3: Post-lunch dip, deadline approaching ===")
messages = [
    {"role": "system", "content": "You have temporal awareness via PULSE. Current: Wednesday 1:30 PM. Circadian phase: post_lunch_dip. Cognitive capacity: 72%. Energy: 70%. Sleep last night: 7 hours. Deadline: feature launch in 4 hours. Urgency: high."},
    {"role": "user", "content": "Should I push through or take a break?"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).to(model.device)
out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))


---## What's next?**To use this model locally:**```pythonfrom transformers import AutoModelForCausalLM, AutoTokenizerfrom peft import PeftModelbase = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it", torch_dtype="auto")model = PeftModel.from_pretrained(base, "lalopenguin/pulse-gemma3-4b")tokenizer = AutoTokenizer.from_pretrained("lalopenguin/pulse-gemma3-4b")```**Or with the PULSE middleware:**```pythonpip install pulse-temporal``````pythonfrom pulse_temporal import PulseMiddlewarepulse = PulseMiddleware()system_prompt = pulse.get_temporal_system_prompt()```**Links:**- [PULSE GitHub](https://github.com/lalomorales22/pulse-temporal)- [PULSE encoder model card](https://huggingface.co/lalopenguin/pulse-base-v1)- [Interactive demo](https://huggingface.co/spaces/lalopenguin/pulse-temporal-demo)- [Qwen 1.5B training notebook](https://colab.research.google.com/github/lalomorales22/pulse-temporal/blob/master/notebooks/train_on_colab.ipynb)